In [26]:
import json
from pathlib import Path

In [27]:
SOURCE_FILE = 'dataset/figma-data/raw/components/components-simple-figma-data.json'
OUTPUT_DIR = 'dataset/figma-data/split/components/simple'

In [28]:
with open(SOURCE_FILE, 'r', encoding='utf-8') as f:
    source_data = json.load(f)

In [29]:
def extract_mockups(source_data: dict) -> tuple[list[dict], dict, dict]:
    """Return (mockups, components, component_sets).

    components/component_sets are the lookup tables Figma stores next to
    'document' in the same nodes[id] wrapper. They're extracted here and
    later saved alongside each split-out mockup, so componentId values
    stay resolvable to a readable name during the cleaning step -
    without this, split mockup files carry only raw, meaningless
    componentId strings with no way to look up what they refer to.
    """
    mockups = []
    components = {}
    component_sets = {}
    nodes = source_data.get('nodes', {})

    for node_id, wrapper in nodes.items():
        doc = wrapper.get('document', {})
        children = doc.get('children', []) or []

        if doc.get('type') == 'SECTION' and children:
           mockups = children
           components = wrapper.get('components', {}) or {}
           component_sets = wrapper.get('componentSets', {}) or {}

    return mockups, components, component_sets

extracted_mockups, components_lookup, component_sets_lookup = extract_mockups(source_data)

print(f'Extracted {len(extracted_mockups)} mockups:')
print(f'Components available for lookup: {len(components_lookup)}')
print(f'Component sets available for lookup: {len(component_sets_lookup)}')

for mockup in extracted_mockups:
    print(f'  - {mockup.get("name")} (type: {mockup.get("type")}, id: {mockup.get("id")})')

Extracted 10 mockups:
Components available for lookup: 36
Component sets available for lookup: 18
  - 10 [Filter] (type: FRAME, id: 4:32)
  - 9 [Todo-Liste] (type: FRAME, id: 4:30)
  - 8 [Suche] (type: FRAME, id: 4:28)
  - 7 [Loading State] (type: FRAME, id: 4:26)
  - 6 [Datei-Upload-Status] (type: FRAME, id: 4:24)
  - 5 [Feedback-Eingabe] (type: FRAME, id: 4:22)
  - 4 [Optionen-Auswahl] (type: FRAME, id: 4:20)
  - 3 [Bewertungs-Eingabe] (type: FRAME, id: 4:18)
  - 2 [Status-Reihe] (type: FRAME, id: 4:16)
  - 1 [Password-Eingabe] (type: FRAME, id: 2:5)


In [30]:
OUTPUT_DIR_PATH = Path(OUTPUT_DIR)
OUTPUT_DIR_PATH.mkdir(parents=True, exist_ok=True)

for mockup in extracted_mockups:
    mockup_name = mockup.get('name')
    mockup_number = mockup_name.split(' ')[0]

    filename = f'{mockup_number}.json'
    file_path = OUTPUT_DIR_PATH / filename

    # Wrap back into the standard Figma REST API shape
    # (nodes[id].document/components/componentSets) instead of writing
    # the bare mockup dict. This keeps the split output identical in
    # shape to a normal Figma export, so the cleaning notebook's
    # extract_document_node()/componentId-resolution logic works
    # unchanged on split files too - no separate code path needed.
    wrapped = {
        'nodes': {
            mockup.get('id'): {
                'document': mockup,
                'components': components_lookup,
                'componentSets': component_sets_lookup,
            }
        }
    }

    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(wrapped, f, indent=4, ensure_ascii=False)

    print(f'Saved mockup "{mockup_name}" to {filename}')

Saved mockup "10 [Filter]" to 10.json
Saved mockup "9 [Todo-Liste]" to 9.json
Saved mockup "8 [Suche]" to 8.json
Saved mockup "7 [Loading State]" to 7.json
Saved mockup "6 [Datei-Upload-Status]" to 6.json
Saved mockup "5 [Feedback-Eingabe]" to 5.json
Saved mockup "4 [Optionen-Auswahl]" to 4.json
Saved mockup "3 [Bewertungs-Eingabe]" to 3.json
Saved mockup "2 [Status-Reihe]" to 2.json
Saved mockup "1 [Password-Eingabe]" to 1.json
